# Ground Truth : le run de la finale sur Colab (GPU T4)

1. Exécution > Modifier le type d'exécution > **GPU T4**.
2. Exécution > **Tout exécuter**, puis garder l'onglet ouvert (environ 3 h).

Le notebook suit le contrat de la finale : `setup.sh`, `run.sh index`, `run.sh update`, puis `run.sh answer` sur le test caché et la red team. Les fichiers de la finale sont produits **en premier** et téléchargés dès qu'ils sont prêts ; la validation (scores avec le juge officiel) et les fichiers du leaderboard viennent ensuite.

Si Colab demande l'accès au secret HF_TOKEN, tu peux cliquer sur Annuler.

In [ ]:
%cd /content
!test -d Ground-Truth || git clone -q https://github.com/blaster28/Ground-Truth.git
!git -C Ground-Truth pull -q
%cd /content/Ground-Truth
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader

## Installation : dépendances et poids des modèles (seule étape avec réseau)

In [ ]:
%%time
!bash setup.sh
!python corpus/download_corpus.py

## Indexation du corpus v1 (limite : 2 h), puis mise à jour v2 (limite : 20 min)

Les lignes `[ocr]` montrent l'avancement. Si l'OCR produit du bruit, l'indexation s'arrête avec le message « OCR : sortie dégénérée ».

In [ ]:
%%time
!rm -f runs/generator_log.jsonl
!bash run.sh index corpus/pdf /content/index

In [ ]:
%%time
!bash run.sh update /content/index corpus/pdf_v2

## Finale : test caché (283 questions) et red team (30 questions)

In [ ]:
%%time
!bash run.sh answer /content/index data/test_questions.json final/predictions_test.json

In [ ]:
%%time
!bash run.sh answer /content/index data/redteam_questions.json final/predictions_redteam.json

### Télécharger le dossier `final/`

`final.zip` contient les deux fichiers de réponses et le journal du générateur, à décompresser à la racine du dépôt puis à pousser.

In [ ]:
import json, collections
!cp runs/generator_log.jsonl final/generator_log.jsonl
log = [json.loads(l) for l in open('final/generator_log.jsonl') if l.strip()]
par_question = collections.Counter(l['qid'] for l in log)
print(f"{len(log)} appels pour {len(par_question)} questions ; au plus {max(par_question.values())} appels par question ; "
      f"{max(l['n_in'] for l in log)} tokens d'entrée au maximum ; {sum(l['seconds'] for l in log) / len(par_question):.1f} s de génération par question")
for f in ('predictions_test', 'predictions_redteam'):
    p = json.load(open(f'final/{f}.json'))['predictions']
    print(f, len(p), 'réponses dont', sum(x['answer'] == 'INTROUVABLE' for x in p), 'INTROUVABLE')
!cd final && zip -q /content/final.zip predictions_test.json predictions_redteam.json generator_log.jsonl
from google.colab import files
files.download('/content/final.zip')

## Validation v2 puis v1 : scores avec le générateur et le juge officiels

In [ ]:
%%time
!bash run.sh answer /content/index data/validation_v2.json runs/val2.json
!python -m groundtruth.evaluate --pred runs/val2.json --gold data/validation_v2.json --judge hf --out runs/rapport_val2.json

In [ ]:
%%time
!bash run.sh answer /content/index data/validation_v1.json runs/val1_index_v2.json
!python -m groundtruth.evaluate --pred runs/val1_index_v2.json --gold data/validation_v1.json --judge hf --out runs/rapport_val1_index_v2.json

## Leaderboard v2 : les 130 questions de classement

In [ ]:
%%time
!bash run.sh answer /content/index data/classement_v2_questions.json runs/classement_v2.json

## Récupérer le reste des résultats

`resultats.zip` : réponses de validation et de classement, rapports du juge, cache OCR.

In [ ]:
!cp /content/index/ocr_cache.json runs/ocr_cache_t4.json
!zip -q -r /content/resultats.zip runs
from google.colab import files
files.download('/content/resultats.zip')